In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze 12: Vessel Identity Enrichment (GFW v3 `/vessels` API — Schema-on-Write All-String)
# MAGIC
# MAGIC - Extracts all distinct vessel IDs (`vessel.id` and `encounter.vessel.id`) from Bronze event JSONL files using explicit all-string `StructType` + `_rescued_data`.
# MAGIC - Fetches missing vessel metadata from GFW v3 `/vessels` in batches of 50 (`MAX_ENRICH_VESSELS = None` for 100% coverage).
# MAGIC - **Enforces Schema-on-Write (All-String):** Parses API responses against the verified GFW `/vessels` all-string `StructType` + `_rescued_data` (`primitivesAsString=true`); quarantines only structurally empty/corrupt records to `silver/_quarantine/schema_drift` and writes schema-enforced all-string JSON to Bronze.
# MAGIC - Logs run metrics (`rows_in`, `rows_out`, `rows_inserted`, `rows_updated`, `quarantined_rows`) to `silver/_audit/pipeline_run_log`.

# COMMAND ----------

import json
import math
import time
from datetime import datetime, timezone
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    StringType,
    StructField,
    StructType,
)

# ── 1. Configuration ─────────────────────────────────────────────────────────
SA_NAME = "stmaritimeks"
BRONZE_ROOT = f"abfss://bronze@{SA_NAME}.dfs.core.windows.net"
SILVER_ROOT = f"abfss://silver@{SA_NAME}.dfs.core.windows.net"
BASE_URL = "https://gateway.api.globalfishingwatch.org/v3"

KV_SCOPE = "maritime-kv"
API_TOKEN = dbutils.secrets.get(scope=KV_SCOPE, key="gfw-api-token")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

BATCH_SIZE = 50
MAX_ENRICH_VESSELS = None  # Fetch 100% of all unenriched vessels across initial + incremental events

QUARANTINE_PATH = f"{SILVER_ROOT}/_quarantine/schema_drift"
AUDIT_LOG_PATH = f"{SILVER_ROOT}/_audit/pipeline_run_log"

# ── 2. Explicit Upfront All-String Schemas (+ _rescued_data) ─────────────────
BRONZE_EVENT_VESSEL_SCHEMA = StructType([
    StructField("id", StringType(), True),
    StructField("vessel", StructType([
        StructField("id", StringType(), True),
    ]), True),
    StructField("encounter", StructType([
        StructField("vessel", StructType([
            StructField("id", StringType(), True),
        ]), True),
    ]), True),
    StructField("_rescued_data", StringType(), True),
])

BRONZE_VESSEL_ALL_STRING_SCHEMA = StructType([
    StructField("dataset", StringType(), True),
    StructField("registryInfoTotalRecords", StringType(), True),
    StructField("registryInfo", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("ssvid", StringType(), True),
        StructField("imo", StringType(), True),
        StructField("shipname", StringType(), True),
        StructField("nShipname", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("flag", StringType(), True),
        StructField("latestVesselInfo", StringType(), True),
        StructField("transmissionDateFrom", StringType(), True),
        StructField("transmissionDateTo", StringType(), True),
        StructField("geartypes", ArrayType(StringType()), True),
        StructField("lengthM", StringType(), True),
        StructField("tonnageGt", StringType(), True),
        StructField("enginePowerKw", StringType(), True),
        StructField("vesselInfoReference", StringType(), True),
        StructField("recordId", StringType(), True),
        StructField("extraFields", ArrayType(StructType([
            StructField("registrySource", StringType(), True),
            StructField("iuuStatus", StructType([
                StructField("value", StringType(), True),
                StructField("dateFrom", StringType(), True),
                StructField("dateFromMask", StringType(), True),
                StructField("dateTo", StringType(), True),
                StructField("dateToMask", StringType(), True),
            ]), True),
            StructField("hasComplianceInfo", StringType(), True),
            StructField("images", ArrayType(StringType()), True),
            StructField("operator", StringType(), True),
            StructField("builtYear", StringType(), True),
            StructField("depthM", StringType(), True),
        ])), True),
    ])), True),
    StructField("registryOwners", ArrayType(StructType([
        StructField("name", StringType(), True),
        StructField("flag", StringType(), True),
        StructField("ssvid", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("dateFrom", StringType(), True),
        StructField("dateTo", StringType(), True),
    ])), True),
    StructField("registryPublicAuthorizations", ArrayType(StructType([
        StructField("dateFrom", StringType(), True),
        StructField("dateTo", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("rfmo", StringType(), True),
        StructField("originalRfmo", StringType(), True),
    ])), True),
    StructField("selfReportedInfo", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("ssvid", StringType(), True),
        StructField("shipname", StringType(), True),
        StructField("nShipname", StringType(), True),
        StructField("flag", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("imo", StringType(), True),
        StructField("transmissionDateFrom", StringType(), True),
        StructField("transmissionDateTo", StringType(), True),
        StructField("sourceCode", ArrayType(StringType()), True),
        StructField("matchFields", StringType(), True),
        StructField("index", StringType(), True),
        StructField("messagesCounter", StringType(), True),
        StructField("positionsCounter", StringType(), True),
        StructField("reference", StringType(), True),
    ])), True),
    StructField("combinedSourcesInfo", ArrayType(StructType([
        StructField("vesselId", StringType(), True),
        StructField("geartypes", ArrayType(StructType([
            StructField("name", StringType(), True),
            StructField("source", StringType(), True),
            StructField("yearFrom", StringType(), True),
            StructField("yearTo", StringType(), True),
        ])), True),
        StructField("shiptypes", ArrayType(StructType([
            StructField("name", StringType(), True),
            StructField("source", StringType(), True),
            StructField("yearFrom", StringType(), True),
            StructField("yearTo", StringType(), True),
        ])), True),
        StructField("bestLengthM", StringType(), True),
        StructField("bestTonnageGt", StringType(), True),
        StructField("bestEnginePowerKw", StringType(), True),
        StructField("maxSpeedKn", StringType(), True),
        StructField("yearFrom", StringType(), True),
        StructField("yearTo", StringType(), True),
    ])), True),
    StructField("_rescued_data", StringType(), True),
])

# COMMAND ----------

# ── 3. Unified Quarantine & Audit Helpers ────────────────────────────────────
def write_schema_drift_quarantine(df_quarantine, layer: str, entity: str, run_id: str) -> int:
    if df_quarantine.isEmpty():
        return 0
    count = df_quarantine.count()
    out = (
        df_quarantine
        .withColumn("quarantine_ts_utc", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("run_id", F.lit(run_id))
        .withColumn("layer", F.lit(layer))
        .withColumn("entity", F.lit(entity))
        .select(
            "quarantine_ts_utc",
            "run_id",
            "layer",
            "entity",
            F.col("_source_file").alias("source_file"),
            F.coalesce(F.col("_rescued_data"), F.lit("EMPTY_VESSEL_RECORD")).alias("rescued_payload"),
        )
    )
    out.write.format("delta").option("mergeSchema", "true").mode("append").save(QUARANTINE_PATH)
    return count


def log_pipeline_audit(
    run_id: str,
    layer: str,
    notebook_name: str,
    entity: str,
    status: str,
    started_at_utc: str,
    rows_in: int,
    rows_out: int,
    rows_inserted: int = 0,
    rows_updated: int = 0,
    quarantined_rows: int = 0,
    duplicates_removed: int = 0,
    error_message: str = None,
):
    audit_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("entity", StringType(), True),
        StructField("status", StringType(), True),
        StructField("started_at_utc", StringType(), True),
        StructField("finished_at_utc", StringType(), True),
        StructField("rows_in", StringType(), True),
        StructField("rows_out", StringType(), True),
        StructField("rows_inserted", StringType(), True),
        StructField("rows_updated", StringType(), True),
        StructField("quarantined_rows", StringType(), True),
        StructField("duplicates_removed", StringType(), True),
        StructField("error_message", StringType(), True),
    ])
    row = [(
        run_id,
        layer,
        notebook_name,
        entity,
        status,
        started_at_utc,
        datetime.now(timezone.utc).isoformat(),
        str(rows_in),
        str(rows_out),
        str(rows_inserted),
        str(rows_updated),
        str(quarantined_rows),
        str(duplicates_removed),
        error_message,
    )]
    (
        spark.createDataFrame(row, schema=audit_schema)
        .write.format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .save(AUDIT_LOG_PATH)
    )

# COMMAND ----------

# ── 4. API Fetch & Schema-on-Write Enforcement ───────────────────────────────
def fetch_vessel_batch(vessel_ids: list):
    url = f"{BASE_URL}/vessels"
    params = [("datasets[0]", "public-global-vessel-identity:latest")]
    for idx, vid in enumerate(vessel_ids):
        params.append((f"ids[{idx}]", vid))

    retries_used = 0
    for attempt in range(1, 6):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if resp.status_code == 200:
            return resp.json().get("entries", []), retries_used
        if resp.status_code in (429, 500, 502, 503, 504):
            retries_used += 1
            retry_after = resp.headers.get("Retry-After")
            wait_sec = int(retry_after) if (retry_after and retry_after.isdigit()) else (2 ** attempt)
            print(f"  [HTTP {resp.status_code}] Retrying in {wait_sec}s (attempt {attempt}/5)...")
            time.sleep(wait_sec)
        else:
            resp.raise_for_status()

    raise RuntimeError(f"Failed after 5 retries fetching vessel batch of size {len(vessel_ids)}")


def enforce_schema_and_write_bronze(entries: list, target_dir: str, source_label: str, run_id: str):
    """
    Enforces Schema-on-Write with all-string StructType + _rescued_data on Serverless compute.
    Validates that the entry contains vessel arrays (selfReportedInfo or registryInfo),
    quarantines only truly empty/corrupt records, and writes all-string JSON to Bronze.
    """
    if not entries:
        return 0, 0

    temp_raw_file = f"{target_dir}_raw_tmp.jsonl"
    body = "\n".join(json.dumps(e) for e in entries)
    dbutils.fs.put(temp_raw_file, body, True)

    try:
        df_parsed = (
            spark.read
            .schema(BRONZE_VESSEL_ALL_STRING_SCHEMA)
            .option("mode", "PERMISSIVE")
            .option("primitivesAsString", "true")
            .option("rescuedDataColumn", "_rescued_data")
            .json(temp_raw_file)
            .withColumn("_source_file", F.lit(source_label))
        )

        has_vessel_data = (
            (F.size(F.coalesce(F.col("selfReportedInfo"), F.array())) > 0)
            | (F.size(F.coalesce(F.col("registryInfo"), F.array())) > 0)
        )

        df_broken = df_parsed.filter(~has_vessel_data)
        q_count = write_schema_drift_quarantine(df_broken, "bronze", "vessels_api_write", run_id)

        df_valid = df_parsed.filter(has_vessel_data).drop("_source_file")
        v_count = df_valid.count()

        if v_count > 0:
            df_valid.write.mode("overwrite").json(target_dir)

        return v_count, q_count
    finally:
        dbutils.fs.rm(temp_raw_file, False)

# COMMAND ----------

# ── 5. Main Execution ────────────────────────────────────────────────────────
def run_vessel_enrichment():
    started_at = datetime.now(timezone.utc).isoformat()
    ingest_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    run_id = f"vessels_enrich_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

    total_candidates = 0
    total_fetched = 0
    quarantined_count = 0

    try:
        # Read all Bronze events using explicit all-string schema + _rescued_data
        events_path = f"{BRONZE_ROOT}/gfw/events/*/*/*/*.jsonl"
        df_events = (
            spark.read
            .schema(BRONZE_EVENT_VESSEL_SCHEMA)
            .option("mode", "PERMISSIVE")
            .option("primitivesAsString", "true")
            .option("rescuedDataColumn", "_rescued_data")
            .json(events_path)
            .withColumn("_source_file", F.col("_metadata.file_path"))
        )

        df_event_broken = df_events.filter(F.col("id").isNull() | (F.trim(F.col("id")) == ""))
        quarantined_count += write_schema_drift_quarantine(df_event_broken, "bronze", "events_vessel_ids", run_id)

        # Collect both primary vessel.id and encounter.vessel.id
        primary_ids = df_events.select(F.col("vessel.id").alias("vessel_id"))
        encounter_ids = df_events.select(F.col("encounter.vessel.id").alias("vessel_id"))
        all_event_vessels = (
            primary_ids.unionByName(encounter_ids)
            .filter(F.col("vessel_id").isNotNull() & (F.trim(F.col("vessel_id")) != ""))
            .distinct()
        )
        total_candidates = all_event_vessels.count()

        # Left-anti join against already-stored Bronze vessels (supports both batch_XXXX/part-*.json and *.jsonl)
        vessels_glob = f"{BRONZE_ROOT}/gfw/vessels/*/*/*"
        try:
            df_existing = (
                spark.read
                .schema(BRONZE_VESSEL_ALL_STRING_SCHEMA)
                .option("mode", "PERMISSIVE")
                .option("primitivesAsString", "true")
                .option("rescuedDataColumn", "_rescued_data")
                .json(vessels_glob)
            )
            existing_sri = df_existing.select(F.explode_outer("selfReportedInfo.id").alias("vessel_id"))
            existing_reg = df_existing.select(F.explode_outer("registryInfo.id").alias("vessel_id"))
            known_vessels = (
                existing_sri.unionByName(existing_reg)
                .filter(F.col("vessel_id").isNotNull() & (F.trim(F.col("vessel_id")) != ""))
                .distinct()
            )
            df_to_fetch = all_event_vessels.join(known_vessels, on="vessel_id", how="left_anti")
        except Exception:
            df_to_fetch = all_event_vessels

        if MAX_ENRICH_VESSELS is not None:
            df_to_fetch = df_to_fetch.limit(MAX_ENRICH_VESSELS)

        new_ids = [r["vessel_id"] for r in df_to_fetch.collect()]
        print(f"Total distinct event vessel IDs: {total_candidates} | New IDs queued for fetch: {len(new_ids)}")

        if not new_ids:
            log_pipeline_audit(
                run_id=run_id,
                layer="bronze",
                notebook_name="12_bronze_vessels_enrichment",
                entity="vessels",
                status="SUCCESS",
                started_at_utc=started_at,
                rows_in=total_candidates,
                rows_out=0,
                rows_inserted=0,
                rows_updated=0,
                quarantined_rows=quarantined_count,
            )
            print("No new vessel IDs to enrich.")
            return

        total_batches = math.ceil(len(new_ids) / BATCH_SIZE)
        manifest_records = []

        for b_idx in range(total_batches):
            batch_ids = new_ids[b_idx * BATCH_SIZE : (b_idx + 1) * BATCH_SIZE]
            batch_num = f"{b_idx + 1:04d}"
            b_started = datetime.now(timezone.utc).isoformat()
            target_dir = (
                f"{BRONZE_ROOT}/gfw/vessels/load_type=incremental/"
                f"ingest_date={ingest_date}/batch_{batch_num}"
            )
            try:
                entries, retries = fetch_vessel_batch(batch_ids)
                valid_written, batch_q = enforce_schema_and_write_bronze(
                    entries, target_dir, f"gfw_v3_vessels_batch_{batch_num}", run_id
                )
                total_fetched += valid_written
                quarantined_count += batch_q
                status = "SUCCESS"
                err_msg = None
                if (b_idx + 1) % 10 == 0 or (b_idx + 1) == total_batches:
                    print(
                        f"[OK] Batch {b_idx + 1}/{total_batches}: wrote {valid_written} "
                        f"all-string vessels (quarantined={batch_q}) -> {target_dir}"
                    )
            except Exception as exc:
                valid_written = 0
                retries = 5
                status = "FAILED"
                err_msg = str(exc)[:300]
                print(f"[ERR] Batch {b_idx + 1}/{total_batches}: {err_msg}")

            manifest_records.append({
                "run_id": run_id,
                "entity": "vessels",
                "dataset": "public-global-vessel-identity:latest",
                "region": f"batch_{batch_num}",
                "window_start": None,
                "window_end": None,
                "load_type": "incremental",
                "source": "gfw_v3_vessels_api",
                "rows_received": valid_written,
                "file_path": target_dir,
                "status": status,
                "retry_count": retries,
                "error_message": err_msg,
                "started_at_utc": b_started,
                "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                "source_url": f"{BASE_URL}/vessels",
                "api_version": "v3",
            })
            time.sleep(0.2)

        manifest_path = f"{BRONZE_ROOT}/_manifest/ingestion_manifest/{run_id}.jsonl"
        dbutils.fs.put(manifest_path, "\n".join(json.dumps(r) for r in manifest_records), True)
        print(f"Manifest written to {manifest_path}")

        log_pipeline_audit(
            run_id=run_id,
            layer="bronze",
            notebook_name="12_bronze_vessels_enrichment",
            entity="vessels",
            status="SUCCESS",
            started_at_utc=started_at,
            rows_in=len(new_ids),
            rows_out=total_fetched,
            rows_inserted=total_fetched,
            rows_updated=0,
            quarantined_rows=quarantined_count,
        )

    except Exception as e:
        log_pipeline_audit(
            run_id=run_id,
            layer="bronze",
            notebook_name="12_bronze_vessels_enrichment",
            entity="vessels",
            status="FAILED",
            started_at_utc=started_at,
            rows_in=total_candidates,
            rows_out=total_fetched,
            rows_inserted=total_fetched,
            rows_updated=0,
            quarantined_rows=quarantined_count,
            error_message=str(e)[:500],
        )
        raise


if __name__ == "__main__":
    run_vessel_enrichment()

In [0]:
def list_bronze_tree(path, depth=0, max_depth=3):
    if depth > max_depth:
        return
    try:
        items = dbutils.fs.ls(path)
    except Exception as e:
        print("  " * depth + f"[ERR] {path}: {e}")
        return
    for item in items:
        if item.isDir():
            print("  " * depth + f"📁 {item.name}")
            list_bronze_tree(item.path, depth + 1, max_depth)

list_bronze_tree("abfss://bronze@stmaritimeks.dfs.core.windows.net/")